<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🕸️</div><div style="color:white;font-size:1.5rem;font-weight:700;">Network Visualiser</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Generate network graphs from structured edge-list data<br><a href="https://ladal.edu.au/net.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>

<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">How to use this tool:</b><ol style="margin:.5rem 0 0 0;padding-left:1.3em;line-height:1.9;"><li>Run <b>Step 1</b> to set up the environment.</li><li>Upload your <code>.xlsx</code> edge-list file to the <b>MyTables</b> folder, then run <b>Step 2</b>.</li><li>In <b>Step 3</b>, select your file and layout, then click <b>Run Analysis</b>.</li><li>Download the network image from the <b>MyOutput</b> folder.</li></ol></div>

<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">&#x1F4D6; <b>This notebook contains code cells.</b> Do not edit the code — just run each step using the <b>Run Step</b> buttons. To run a cell: click on it and press <b>Shift + Enter</b>, or use the <b>&#x25B6; Run</b> button in the toolbar above.</div>

<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 1</span><br><b style="font-size:1rem;">&#x2699;&#xFE0F; Set up the environment — run this first</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;"><b>Run the cell below once</b> to load all the helper functions this notebook needs. Click on the cell, then press <b>Shift + Enter</b>. You should see a green <em>Setup complete</em> message when it finishes.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 1 — environment setup — run this cell, do not edit it</em></div>

In [ ]:
# Setup: load display helpers and shared functions
# Run this cell once — then proceed to the steps below.

suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))
suppressPackageStartupMessages(library(ipywidgets))

# ── Colour-coded feedback boxes ──────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:", bg,
    ";border-left:4px solid ", border,
    ";border-radius:6px;padding:10px 15px;margin:.4rem 0;",
    "font-family:sans-serif;font-size:.9rem;\">",
    if (nzchar(icon)) paste0(icon, " ") else "",
    msg, "</div>"))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar ─────────────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    "<div style=\"font-family:sans-serif;font-size:.85rem;margin:.3rem 0;\">",
    "<span style=\"color:#51247a;font-weight:600;\">", label, "</span>",
    "<div style=\"background:#e8e4f0;border-radius:4px;height:8px;margin-top:3px;\">",
    "<div style=\"background:#51247a;width:", pct,
    "%;height:8px;border-radius:4px;\"></div></div></div>"))
}

# ── Load plain-text files from a folder ──────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in ", folder,
         ". Please upload your files first.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts to MyOutput ───────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save data frame as Excel ─────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Setup complete. Follow the steps below.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 2</span><br><b style="font-size:1rem;">&#x1F4C2; Upload your edge-list Excel file to MyTables</b></div>

**Input format:** Your Excel file needs three columns: `from` (start node), `to` (end node), and `weight` or `s` (edge strength — use `1` for unweighted).

<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Look at the <b>file browser panel</b> on the left side of the screen.</li><li>Double-click the <b><code>MyTables</code></b> folder to open it.</li><li><b>Drag and drop</b> your <code>.xlsx</code> files into that folder.</li><li>When done, click the <b>Run Step</b> button below.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.xlsx</code> files are accepted. You can upload multiple files at once.</p></div>

<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 3</span><br><b style="font-size:1rem;">&#x1F4CA; Configure and run the analysis</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Set your options using the controls below, then click the green <b>Run Analysis</b> button. Results will appear directly below the button and will also be saved to the <b>MyOutput</b> folder.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 3 — analysis — run this cell, do not edit it</em></div>

In [ ]:
# Configure the network and click Run Analysis.
suppressPackageStartupMessages(library(readxl))
suppressPackageStartupMessages(library(igraph))
suppressPackageStartupMessages(library(ggraph))
suppressPackageStartupMessages(library(tidygraph))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(scales))

available_files <- tryCatch(
  list.files("notebooks/MyTables", pattern="\\.xlsx$"),
  error=function(e) character(0))
if (length(available_files)==0) available_files <- c("edgelist.xlsx")

w_file   <- ipywidgets::Dropdown(
  options=as.list(available_files),
  description="Excel file:",
  style=list(description_width="120px"),
  layout=ipywidgets::Layout(width="340px"))
w_layout <- ipywidgets::Dropdown(
  options=list("Fruchterman-Reingold"="fr","Kamada-Kawai"="kk",
               "Circle"="circle","Star"="star"),
  value="fr", description="Graph layout:",
  style=list(description_width="120px"),
  layout=ipywidgets::Layout(width="300px"))
w_minw   <- ipywidgets::FloatSlider(
  value=1, min=0, max=10, step=0.5,
  description="Min. edge weight:",
  style=list(description_width="150px"),
  layout=ipywidgets::Layout(width="420px"))
w_btn    <- ipywidgets::Button(
  description="  Run Analysis",
  button_style="success", icon="share-alt",
  layout=ipywidgets::Layout(width="180px", height="38px"))
w_out    <- ipywidgets::Output()

ipywidgets::display(ipywidgets::VBox(list(
  ipywidgets::HTML(paste0(
    "<div style=\"background:#f0f7ff;border:1px solid #4085C6;",
    "border-radius:6px;padding:14px 18px;margin:.5rem 0;\">",
    "<b style=\"color:#2a5ea8;\">Configure the network graph:</b></div>")),
  w_file, w_layout, w_minw, w_btn, w_out
)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      .prog("Loading data...", 20)
      path  <- file.path("notebooks/MyTables", w_file$value)
      if (!file.exists(path)) stop("File not found: ", path,
        ". Please upload your Excel file to the MyTables folder.")
      edges <- readxl::read_xlsx(path)
      names(edges) <- tolower(names(edges))
      if ("s" %in% names(edges) && !"weight" %in% names(edges))
        edges <- dplyr::rename(edges, weight=s)
      if (!"weight" %in% names(edges)) edges$weight <- 1
      edges <- dplyr::filter(edges, weight >= w_minw$value)
      ok(paste0("Loaded <b>", nrow(edges), " edges</b> | <b>",
                length(unique(c(edges$from, edges$to))), " nodes</b>"))
      .prog("Building graph...", 55)
      g  <- igraph::graph_from_data_frame(edges, directed=FALSE)
      tg <- tidygraph::as_tbl_graph(g) |>
        tidygraph::activate(nodes) |>
        dplyr::mutate(degree=tidygraph::centrality_degree(),
                      lsize=scales::rescale(degree, to=c(3,8)))
      .prog("Rendering network...", 80)
      p <- ggraph::ggraph(tg, layout=w_layout$value) +
        ggraph::geom_edge_link(aes(width=weight, alpha=weight),
                               colour="gray60", show.legend=FALSE) +
        ggraph::scale_edge_width(range=c(0.3,2.5)) +
        ggraph::scale_edge_alpha(range=c(0.2,0.8)) +
        ggraph::geom_node_point(aes(size=degree), colour="#51247a",
                                alpha=0.85, show.legend=FALSE) +
        ggraph::geom_node_label(aes(label=name, size=lsize),
                                colour="white", fill="#51247a", fontface="bold",
                                label.padding=unit(0.15,"lines"),
                                label.size=0, show.legend=FALSE) +
        ggraph::theme_graph(base_family="sans") +
        labs(title=tools::file_path_sans_ext(w_file$value))
      print(p)
      .prog("Saving...", 95)
      dir.create("notebooks/MyOutput", showWarnings=FALSE, recursive=TRUE)
      ggsave("notebooks/MyOutput/network_graph.png", plot=p,
             bg="white", width=10, height=8, dpi=200)
      .prog("Done.", 100)
      ok("Saved <b>network_graph.png</b> to MyOutput. Right-click → Download.")
    }, error=function(e) err(conditionMessage(e)))
  })
})


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 99</span><br><b style="font-size:1rem;">&#x2B07;&#xFE0F; Download your results</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Your results have been saved to the <b>MyOutput</b> folder. To download: open the <b>file browser panel</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> on a file and choose <b>Download</b>.</div>

---

### Citation

Schweinberger, Martin. (2025). *LADAL Network Visualiser*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025network,
  author = {Schweinberger, Martin},
  title  = {LADAL Network Visualiser},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()